In [2]:
# Membuat direktori raw dan processed di HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/raw
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/processed

# Menampilkan struktur direktori untuk verifikasi
!hdfs dfs -ls -R /user/mahasiswa/ecommerce

drwxr-xr-x   - azizi supergroup          0 2026-09-09 22:47 /user/mahasiswa/ecommerce/processed
drwxr-xr-x   - azizi supergroup          0 2026-09-09 22:47 /user/mahasiswa/ecommerce/raw


In [3]:
# Mengunggah ketiga file CSV transaksi cabang ke direktori raw HDFS
!hdfs dfs -put transaksi_magelang.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/mahasiswa/ecommerce/raw/

# Verifikasi isi direktori beserta ukuran berkas
!hdfs dfs -ls -h /user/mahasiswa/ecommerce/raw

Found 3 items
-rw-r--r--   1 azizi supergroup     12.0 K 2026-09-09 22:47 /user/mahasiswa/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 azizi supergroup     11.9 K 2026-09-09 22:47 /user/mahasiswa/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 azizi supergroup     12.4 K 2026-09-09 22:47 /user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv


In [4]:
import io
import subprocess
import pandas as pd


# Fungsi membaca berkas CSV langsung dari HDFS menggunakan command line interface
def read_hdfs_csv(hdfs_path):
    result = subprocess.run(
        ["hdfs", "dfs", "-cat", hdfs_path],
        capture_output=True,
        text=True,
        check=True,
    )
    return pd.read_csv(io.StringIO(result.stdout))


# Membaca ketiga berkas dari HDFS
df_magelang = read_hdfs_csv(
    "/user/mahasiswa/ecommerce/raw/transaksi_magelang.csv"
)
df_yogya = read_hdfs_csv(
    "/user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv"
)
df_semarang = read_hdfs_csv(
    "/user/mahasiswa/ecommerce/raw/transaksi_semarang.csv"
)

# Menggabungkan menjadi satu DataFrame
df_gabungan = pd.concat(
    [df_magelang, df_yogya, df_semarang], ignore_index=True
)

# Menampilkan statistik jumlah transaksi per kota
print(df_gabungan["kota"].value_counts())

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [5]:
# 1. Menambahkan kolom total_pendapatan
df_gabungan["total_pendapatan"] = (
    df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
)

# 2. Membuat ringkasan total pendapatan per kota dan per kategori
df_ringkasan = (
    df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
)

# Tampilkan tabel ringkasan
display(df_ringkasan)

# 3. Menyimpan kedua hasil ke disk lokal
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

# Mengunggah berkas hasil ke folder processed di HDFS
!hdfs dfs -put -f data_gabungan_bersih.csv /user/mahasiswa/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/mahasiswa/ecommerce/processed/

# Verifikasi keberhasilan unggah di direktori processed
!hdfs dfs -ls -h /user/mahasiswa/ecommerce/processed

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


Found 2 items
-rw-r--r--   1 azizi supergroup     40.3 K 2026-09-09 22:47 /user/mahasiswa/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 azizi supergroup        530 2026-09-09 22:48 /user/mahasiswa/ecommerce/processed/ringkasan_kota_kategori.csv


Pemisahan antara direktori raw dan processed pada HDFS merupakan salah satu praktik terbaik dalam arsitektur Big Data yang menganut prinsip data lake. Keuntungan utamanya adalah menjaga integritas data mentah sebagai single source of truth. Data pada folder raw disimpan sesuai bentuk aslinya dan diperlakukan sebagai data yang tak boleh diubah. Jika terdapat kesalahan algoritma, bug pada proses pembersihan, atau perubahan kebutuhan pipeline di masa mendatang, kita dapat melakukan pemrosesan ulang dari data sumber tanpa risiko kehilangan data awal. Pemisahan ini juga menyederhanakan tata kelola data, pengaturan hak akses keamanan, serta mencegah tim analisis mengolah data mentah yang belum tervalidasi.